In [4]:
import tvm
from tvm import relax
from tvm.relax.frontend.torch import from_exported_program

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import torchvision.models as models
from torchvision.io import read_image
from torch.export import export

import onnx
import os

# Resnet model

In [7]:
from torchvision.models.resnet import ResNet34_Weights, resnet34
torch_model = resnet34(weights=ResNet34_Weights.DEFAULT).eval()

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /home/leobraginski/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100.0%


IRModule

In [8]:
# Give an example argument to torch.export
example_args = (torch.randn(1, 3, 224, 224, dtype=torch.float32),)

# Convert the model to IRModule
with torch.no_grad():
    exported_program = export(torch_model, example_args)
    mod = from_exported_program(exported_program, keep_params_as_input=True)

mod, params = relax.frontend.detach_params(mod)
mod.show()

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


## Optimize model

In [9]:
TOTAL_TRIALS = 512
MAX_TRIALS_PER_TASK = 16 
target = tvm.target.Target({
    "kind": "llvm",
    "mtriple": "riscv64-linux-gnu",
    "mcpu": "generic-rv64",
    "mattr": ["+64bit", "+m", "+a", "+f", "+d", "+c", "+v"],
    "num-cores": 8,
})
work_dir = "tuning_logs"

mod = relax.get_pipeline(
    "static_shape_tuning",
    target=target,
    work_dir=work_dir,
    total_trials=TOTAL_TRIALS,
    max_trials_per_task=MAX_TRIALS_PER_TASK,
)(mod)

# Only show the main function
mod["main"].show()

2026-03-31 16:28:07 [INFO] Logging directory: tuning_logs/logs
2026-03-31 16:28:19 [INFO] LocalBuilder: max_workers = 8
2026-03-31 16:28:21 [INFO] LocalRunner: max_workers = 1
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #0: "transpose"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #1: "fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #2: "fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_add3_relu1"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #3: "max_pool2d"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #4: "fused_matmul_add13"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #5: "fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_relu1"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #6: "fused_conv2d2_subtrac

[16:28:22] /home/leobraginski/Desktop/TVM-RVV_optimized_operators/tvm/src/meta_schedule/schedule_rule/apply_custom_rule.cc:60: Warning: Unknown schedule rule "meta_schedule.pool_max" for target keys "["cpu"]". Checked ffi::Functions:
  meta_schedule.cpu.meta_schedule.pool_max


2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #9: "fused_conv2d7_subtract3_divide3_expand_dims2_multiply3_expand_dims2_add8"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #10: "fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_add12_relu4"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #11: "fused_conv2d6_subtract3_divide3_expand_dims2_multiply3_expand_dims2_add8_add9_relu3"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #12: "reshape"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #13: "fused_conv2d3_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_add6_relu2"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #14: "fused_conv2d5_subtract3_divide3_expand_dims2_multiply3_expand_dims2_add8_relu3"
2026-03-31 16:28:22 [INFO] [task_scheduler.cc:168] Initializing Task #15: "mean"
2026-03-31 16:28:23 [INFO] [task_scheduler.cc:168]

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,transpose,1,1,N/A,N/A,N/A,0,
1,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,231336448,2,N/A,N/A,N/A,0,
2,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_add3_relu1,232415232,3,N/A,N/A,N/A,0,
3,max_pool2d,1806336,1,N/A,N/A,N/A,0,
4,fused_matmul_add13,1025000,1,N/A,N/A,N/A,0,
5,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_relu1,232214528,3,N/A,N/A,N/A,0,
6,fused_conv2d2_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_relu2,116107264,1,N/A,N/A,N/A,0,
7,fused_conv2d10_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11,12945408,1,N/A,N/A,N/A,0,
8,fused_conv2d4_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5,13246464,1,N/A,N/A,N/A,0,
9,fused_conv2d7_subtract3_divide3_expand_dims2_multiply3_expand_dims2_add8,13045760,1,N/A,N/A,N/A,0,


2026-03-31 16:28:23 [DEBUG] [task_scheduler.cc:327] 
 ID |                                                                                  Name |      FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                                                                             transpose |         1 |      1 |            N/A |          N/A |                   N/A |      0 |      
  1 |       fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4 | 231336448 |      2 |            N/A |          N/A |                   N/A |      0 |      
  2 |     fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_add3_relu1 | 232415232 |      3 |            N/A |          N/A |                   N/A |      0 |      
  3 |            

...

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,transpose,1,1,N/A,N/A,N/A,1,
1,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,231336448,2,N/A,N/A,N/A,16,Y
2,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_add3_relu1,232415232,3,N/A,N/A,N/A,16,Y
3,max_pool2d,1806336,1,N/A,N/A,N/A,16,Y
4,fused_matmul_add13,1025000,1,N/A,N/A,N/A,16,Y
5,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_relu1,232214528,3,N/A,N/A,N/A,16,Y
6,fused_conv2d2_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_relu2,116107264,1,N/A,N/A,N/A,16,Y
7,fused_conv2d10_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11,12945408,1,N/A,N/A,N/A,16,Y
8,fused_conv2d4_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5,13246464,1,N/A,N/A,N/A,16,Y
9,fused_conv2d7_subtract3_divide3_expand_dims2_multiply3_expand_dims2_add8,13045760,1,N/A,N/A,N/A,16,Y


2026-03-31 16:33:59 [DEBUG] [task_scheduler.cc:327] 
 ID |                                                                                  Name |      FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                                                                             transpose |         1 |      1 |            N/A |          N/A |                   N/A |      1 |      
  1 |       fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4 | 231336448 |      2 |            N/A |          N/A |                   N/A |     16 |    Y 
  2 |     fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_add3_relu1 | 232415232 |      3 |            N/A |          N/A |                   N/A |     16 |    Y 
  3 |            

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,transpose,1,1,N/A,N/A,N/A,1,Y
1,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,231336448,2,N/A,N/A,N/A,16,Y
2,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_add3_relu1,232415232,3,N/A,N/A,N/A,16,Y
3,max_pool2d,1806336,1,N/A,N/A,N/A,16,Y
4,fused_matmul_add13,1025000,1,N/A,N/A,N/A,16,Y
5,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_relu1,232214528,3,N/A,N/A,N/A,16,Y
6,fused_conv2d2_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_relu2,116107264,1,N/A,N/A,N/A,16,Y
7,fused_conv2d10_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11,12945408,1,N/A,N/A,N/A,16,Y
8,fused_conv2d4_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5,13246464,1,N/A,N/A,N/A,16,Y
9,fused_conv2d7_subtract3_divide3_expand_dims2_multiply3_expand_dims2_add8,13045760,1,N/A,N/A,N/A,16,Y



Total trials: 0
Total latency (us): 0

2026-03-31 16:33:59 [DEBUG] [task_scheduler.cc:327] 
 ID |                                                                                  Name |      FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                                                                             transpose |         1 |      1 |            N/A |          N/A |                   N/A |      1 |    Y 
  1 |       fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4 | 231336448 |      2 |            N/A |          N/A |                   N/A |     16 |    Y 
  2 |     fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_add3_relu1 | 232415232 |      3 |            N/A |          N/A |                 

[16:33:59] /home/leobraginski/Desktop/TVM-RVV_optimized_operators/tvm/src/relax/transform/meta_schedule.cc:92: Warning: Creating JSONDatabase. Workload at: tuning_logs/database_workload.json, Tuning records at: tuning_logs/database_tuning_record.json
